In [1]:
# Instalaciones necesarias
!pip install shap lime spacy matplotlib-venn
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 73.2 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Librerías comunes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import pickle
import joblib
import shap
import spacy
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from lime.lime_text import LimeTextExplainer
from tqdm import tqdm
from matplotlib_venn import venn2, venn3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

Usando: cuda


In [3]:
# Cargar dataset
test_df = pd.read_csv('/kaggle/input/datasets/victorvalenzueladiaz/test-final-2/test_final.csv')
X_test = test_df['Headline_Text'].astype(str).tolist()
y_test = test_df['Category'].astype(str).str.strip().str.capitalize().map({'False':0,'True':1}).tolist()

In [4]:
# Cargar BETO
beto_model = BertForSequenceClassification.from_pretrained(
    '/kaggle/input/models/victorvalenzueladiaz/beto/pytorch/default/1/modelo_beto_final'
).to(device)
beto_tokenizer = BertTokenizer.from_pretrained(
    '/kaggle/input/models/victorvalenzueladiaz/beto/pytorch/default/1/modelo_beto_final'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
# Cargar BERTIN RoBERTa
roberta_model = RobertaForSequenceClassification.from_pretrained(
    '/kaggle/input/models/victorvalenzueladiaz/roberta/pytorch/default/1/modelo_roberta_final'
).to(device)
roberta_tokenizer = RobertaTokenizer.from_pretrained(
    '/kaggle/input/models/victorvalenzueladiaz/roberta/pytorch/default/1/modelo_roberta_final'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [6]:
# Cargar vocabulario y embedding_matrix (BiGRU+CNN y BiGRU base)
with open('/kaggle/input/models/victorvalenzueladiaz/bigrucnnv2/pytorch/default/1/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

embedding_matrix = torch.load(
    '/kaggle/input/models/victorvalenzueladiaz/bigrucnnv2/pytorch/default/1/embedding_matrix.pt',
    map_location='cpu'
)

In [7]:
# Definición clase BiGRU base
class BiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx, embedding_matrix):
        super(BiGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight.data.copy_(embedding_matrix)
        self.embedding.weight.requires_grad = False
        self.bigru = nn.GRU(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.bigru(x)
        h_cat = torch.cat((h[0], h[1]), dim=1)
        return self.fc(h_cat)

# Cargar BiGRU base
# AJUSTA hidden_dim según los mejores hiperparámetros obtenidos en tu entrenamiento
bigru_base_model = BiGRU(
    vocab_size=len(vocab),
    embed_dim=embedding_matrix.shape[1],
    hidden_dim=128,  # este nuevo checkpoint usa 128
    num_classes=2,
    pad_idx=vocab['<pad>'],
    embedding_matrix=embedding_matrix
).to(device)

bigru_base_model.load_state_dict(torch.load(
    '/kaggle/input/models/victorvalenzueladiaz/bigrubasenew/pytorch/default/1/best_bigru.pt',
    map_location=device
))
bigru_base_model.eval()
print('BiGRU base cargado')

BiGRU base cargado


In [8]:
# Definición clase BiGRU+CNN v2
class CNN_BiGRU_v2(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx, embedding_matrix, num_filters=100, dropout_rate=0.3):
        super(CNN_BiGRU_v2, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout_rate)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in [3, 5, 7]
        ])

        self.bigru = nn.GRU(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.attention = nn.Linear(hidden_dim*2, 1)
        self.fc = nn.Linear(len(self.convs)*num_filters + hidden_dim*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)
        cnn_outs = [torch.max(torch.relu(conv(x.transpose(1,2))), dim=2)[0] for conv in self.convs]
        cnn_out = torch.cat(cnn_outs, dim=1)
        gru_out, _ = self.bigru(x)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        attn_out = torch.sum(gru_out * attn_weights, dim=1)
        combined = torch.cat((cnn_out, attn_out), dim=1)
        combined = self.dropout(combined)
        return self.fc(combined)


# Cargar BiGRU+CNN v2 con hiperparámetros del checkpoint
bigru_cnn_model = CNN_BiGRU_v2(
    vocab_size=len(vocab),
    embed_dim=embedding_matrix.shape[1],
    hidden_dim=512,
    num_classes=2,
    pad_idx=vocab['<pad>'],
    embedding_matrix=embedding_matrix,
    num_filters=100,
    dropout_rate=0.3
).to(device)

bigru_cnn_model.load_state_dict(torch.load(
    '/kaggle/input/models/victorvalenzueladiaz/bigrucnnv2best/pytorch/default/1/best_cnn_bigru_v2_best.pt',
    map_location=device
))
bigru_cnn_model.eval()
print('BiGRU+CNN v2 cargado')


BiGRU+CNN v2 cargado


In [9]:
# Cargar Regresion Logistica
logreg_model = joblib.load('/kaggle/input/models/victorvalenzueladiaz/logreg/scikitlearn/default/1/log_reg_model.pkl')
logreg_vectorizer = joblib.load('/kaggle/input/models/victorvalenzueladiaz/logreg/scikitlearn/default/1/tfidf_vectorizer.pkl')

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.6.1. This might lead to breaking c

In [10]:
# Cargar SVM
svm_model = joblib.load('/kaggle/input/models/victorvalenzueladiaz/svm/scikitlearn/default/1/svm_model.pkl')
svm_vectorizer = joblib.load('/kaggle/input/models/victorvalenzueladiaz/logreg/scikitlearn/default/1/tfidf_vectorizer.pkl')


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:
# Funciones auxiliares de tokenizacion para BiGRU
def tokenize_text(text):
    return text.lower().split()

def text_to_tensor(texts, vocab, max_len=100):
    sequences = []
    for t in texts:
        tokens = tokenize_text(t)
        ids = [vocab.get(tok, vocab.get('<unk>', 0)) for tok in tokens]
        if len(ids) < max_len:
            ids += [vocab['<pad>']] * (max_len - len(ids))
        else:
            ids = ids[:max_len]
        sequences.append(ids)
    return torch.tensor(sequences, dtype=torch.long)

In [12]:
# Funciones de prediccion con batching

def predict_transformer(model, tokenizer, texts, batch_size=32, max_len=512):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encodings = tokenizer(batch, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
        encodings = {k: v.to(device) for k,v in encodings.items()}
        with torch.no_grad():
            outputs = model(**encodings)
        probs = torch.softmax(outputs.logits, dim=1).detach().cpu().numpy()
        all_probs.extend(probs[:,1])
    return np.array(all_probs)

def predict_bigru(model, texts, vocab, max_len=100, batch_size=64):
    model.eval()
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        X_tensor = text_to_tensor(batch, vocab, max_len).to(device)
        with torch.no_grad():
            outputs = model(X_tensor)
            probs = torch.softmax(outputs, dim=1)[:,1].cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

def predict_sklearn(model, vectorizer, texts, batch_size=1000):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        X_vec = vectorizer.transform(batch)
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_vec)[:,1]
        elif hasattr(model, 'decision_function'):
            scores = model.decision_function(X_vec)
            probs = 1 / (1 + np.exp(-scores))
        else:
            raise ValueError(f'Modelo {type(model)} no soporta predict_proba ni decision_function')
        all_probs.extend(probs)
    return np.array(all_probs)

def predict_bigru_array(model, arrays, batch_size=64):
    all_probs = []
    for i in range(0, len(arrays), batch_size):
        batch = arrays[i:i+batch_size]
        X_tensor = torch.tensor(batch, dtype=torch.long).to(device)
        with torch.no_grad():
            outputs = model(X_tensor)
            probs = torch.softmax(outputs, dim=1)[:,1].cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)

In [13]:
# Diccionario de modelos y preprocesadores
modelos = {
    'BETO': lambda x: predict_transformer(beto_model, beto_tokenizer, x, batch_size=32),
    'RoBERTa': lambda x: predict_transformer(roberta_model, roberta_tokenizer, x, batch_size=32),
    'BiGRU': lambda x: predict_bigru(bigru_base_model, x, vocab, batch_size=64),
    'BiGRU+CNN': lambda x: predict_bigru(bigru_cnn_model, x, vocab, batch_size=64),
    'LogReg': lambda x: predict_sklearn(logreg_model, logreg_vectorizer, x, batch_size=1000),
    'SVM': lambda x: predict_sklearn(svm_model, svm_vectorizer, x, batch_size=1000)
}

tokenizers = {
    'BETO': beto_tokenizer,
    'RoBERTa': roberta_tokenizer,
    'BiGRU': vocab,
    'BiGRU+CNN': vocab,
    'LogReg': logreg_vectorizer,
    'SVM': svm_vectorizer
}

# Modelos PyTorch para BiGRU (necesario para predict_bigru_array en SHAP)
modelos_pytorch = {
    'BiGRU': bigru_base_model,
    'BiGRU+CNN': bigru_cnn_model
}

# Umbrales optimos por modelo
umbrales_optimos = {
    'BETO': 0.0238,
    'RoBERTa': 0.0030,
    'BiGRU': 0.6400,
    'BiGRU+CNN': 0.0750,
    'LogReg': 0.5232,
    'SVM': -0.1151
}

 

In [14]:
# Funcion auxiliar para grafico SHAP consistente
def _plot_shap(sorted_tokens, model_name, idx, text, label, threshold, model_fn, suffix=''):
    nombres = [t for t,_ in sorted_tokens]
    valores = [v for _,v in sorted_tokens]
    colores = ['crimson' if v>0 else 'steelblue' for v in valores]

    prob = model_fn([text])[0]
    pred = int(prob >= threshold)

    plt.figure(figsize=(10,8))
    plt.barh(range(len(nombres)), valores, color=colores)
    plt.yticks(range(len(nombres)), nombres)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Impacto SHAP promedio')
    plt.title(f'SHAP Local - {model_name} ejemplo {idx}\npred={pred}, real={label}, umbral={threshold}')
    plt.legend(handles=[
        plt.Line2D([0], [0], color='crimson', lw=4, label='Contribucion positiva (hacia Verdadera)'),
        plt.Line2D([0], [0], color='steelblue', lw=4, label='Contribucion negativa (hacia Falsa)')
    ])
    plt.tight_layout()
    plt.savefig(f'shap_{model_name}_local_example{idx}{suffix}.png', dpi=300)
    plt.close()

In [15]:
# SHAP Global para todos los modelos
def shap_global(model_fn, preprocessor, texts, model_name, sample_size=20, top_n=20):
    sample_texts = texts[:sample_size]

    # Caso Transformer (tokenizer HuggingFace)
    if preprocessor is not None and hasattr(preprocessor, 'encode'):
        def wrapped_fn(x):
            if isinstance(x, np.ndarray):
                x = x.tolist()
            return model_fn([str(t) for t in x])

        explainer = shap.Explainer(wrapped_fn, shap.maskers.Text(preprocessor))
        shap_values = explainer([str(t)[:2000] for t in sample_texts])

        shap_vals_class1 = shap_values[:,:,1] if shap_values.values.ndim == 3 else shap_values

        token_vals = {}
        for i in range(len(sample_texts)):
            for t,v in zip(shap_vals_class1[i].data, shap_vals_class1[i].values):
                tok = t.strip()
                if tok and tok not in ['[CLS]','[SEP]','<s>','</s>','<pad>']:
                    token_vals.setdefault(tok,[]).append(v)

        token_means = {t:np.mean(v) for t,v in token_vals.items()}
        sorted_tokens = sorted(token_means.items(), key=lambda x:abs(x[1]), reverse=True)[:top_n]

    # Caso clasico (vectorizador sklearn)
    elif preprocessor is not None and hasattr(preprocessor, 'transform'):
        background = preprocessor.transform(sample_texts[:50])
        test_data = preprocessor.transform(sample_texts[:20])

        if model_name == 'LogReg':
            explainer = shap.LinearExplainer(logreg_model, background, feature_names=preprocessor.get_feature_names_out())
        elif model_name == 'SVM':
            explainer = shap.LinearExplainer(svm_model, background, feature_names=preprocessor.get_feature_names_out())
        else:
            raise ValueError(f'Modelo clasico no reconocido: {model_name}')

        shap_values = explainer(test_data)
        feature_names = preprocessor.get_feature_names_out()
        shap_means = np.mean(shap_values.values, axis=0)
        sorted_idx = np.argsort(np.abs(shap_means))[::-1][:top_n]
        sorted_tokens = [(feature_names[i], shap_means[i]) for i in sorted_idx]

    # Caso BiGRU (vocabulario dict)
    elif isinstance(preprocessor, dict):
        pytorch_model = modelos_pytorch[model_name]

        def vectorize_texts(txts, max_len=100):
            seqs = []
            for text in txts:
                tokens = text.split()
                ids = [preprocessor.get(tok, 0) for tok in tokens]
                ids = ids[:max_len]
                ids += [0] * (max_len - len(ids))
                seqs.append(ids)
            return np.array(seqs)

        background = vectorize_texts(sample_texts[:50])
        test_data = vectorize_texts(sample_texts[:20])

        explainer = shap.KernelExplainer(
            lambda x: predict_bigru_array(pytorch_model, x), background
        )
        shap_values = explainer.shap_values(test_data)

        id2word = {v: k for k, v in preprocessor.items()}
        token_vals = {}
        for i in range(test_data.shape[0]):
            for j in range(test_data.shape[1]):
                tok_id = test_data[i, j]
                if tok_id != 0:
                    tok = id2word.get(tok_id, '<unk>')
                    val = shap_values[i][j]
                    if isinstance(val, np.ndarray) and val.shape[-1] == 2:
                        val = val[1]
                    token_vals.setdefault(tok, []).append(float(val))

        token_means = {t: np.mean(v) for t,v in token_vals.items()}
        sorted_tokens = sorted(token_means.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]

    else:
        raise ValueError(f'Preprocesador no reconocido para {model_name}')

    # Grafico
    nombres = [t for t,_ in sorted_tokens]
    valores = [v for _,v in sorted_tokens]
    colores = ['crimson' if v>0 else 'steelblue' for v in valores]

    plt.figure(figsize=(10,8))
    plt.barh(range(len(nombres)), valores, color=colores)
    plt.yticks(range(len(nombres)), nombres)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Impacto SHAP promedio')
    plt.title(f'Top {top_n} tokens globales - {model_name}')
    plt.legend(handles=[
        plt.Line2D([0], [0], color='crimson', lw=4, label='Contribucion positiva (hacia Verdadera)'),
        plt.Line2D([0], [0], color='steelblue', lw=4, label='Contribucion negativa (hacia Falsa)')
    ])
    plt.tight_layout()
    plt.savefig(f'shap_{model_name}_global.png', dpi=300)
    plt.close()
    print(f'Guardado: shap_{model_name}_global.png')

In [16]:
def shap_local(model_fn, preprocessor, texts, labels, indices, model_name, threshold=0.5, top_n=15):
    # Caso Transformer
    if preprocessor is not None and hasattr(preprocessor, 'encode'):
        def wrapped_fn(x):
            if isinstance(x, np.ndarray):
                x = x.tolist()
            return model_fn([str(t) for t in x])
        explainer = shap.Explainer(wrapped_fn, shap.maskers.Text(preprocessor))
        for idx in indices:
            text = str(texts[idx])[:2000]
            shap_values = explainer([text])
            if shap_values.values.ndim == 3:
                tokens = shap_values.data[0]
                values = shap_values.values[0,:,1]
            else:
                tokens = shap_values.data[0]
                values = shap_values.values[0]
            token_means = {t: v for t,v in zip(tokens, values)
                           if t not in ['[CLS]','[SEP]','<s>','</s>','<pad>']}
            sorted_tokens = sorted(token_means.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
            _plot_shap(sorted_tokens, model_name, idx, texts[idx], labels[idx], threshold, model_fn)

    # Caso clasico
    elif preprocessor is not None and hasattr(preprocessor, 'transform'):
        background = preprocessor.transform(texts[:50])
        test_data = preprocessor.transform([texts[i] for i in indices])
        if model_name == 'LogReg':
            explainer = shap.LinearExplainer(logreg_model, background, feature_names=preprocessor.get_feature_names_out())
        elif model_name == 'SVM':
            explainer = shap.LinearExplainer(svm_model, background, feature_names=preprocessor.get_feature_names_out())
        else:
            raise ValueError(f'Modelo clasico no reconocido: {model_name}')
        shap_values = explainer(test_data)
        feature_names = preprocessor.get_feature_names_out()
        for i, idx in enumerate(indices):
            shap_vals = shap_values.values[i]
            token_means = {feature_names[j]: shap_vals[j] for j in range(len(feature_names))}
            sorted_tokens = sorted(token_means.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
            _plot_shap(sorted_tokens, model_name, idx, texts[idx], labels[idx], threshold, model_fn)

    # Caso BiGRU
    elif isinstance(preprocessor, dict):
        pytorch_model = modelos_pytorch[model_name]
        def vectorize_texts(txts, max_len=100):
            seqs = []
            for text in txts:
                tokens = text.split()
                ids = [preprocessor.get(tok, 0) for tok in tokens]
                ids = ids[:max_len]
                ids += [0] * (max_len - len(ids))
                seqs.append(ids)
            return np.array(seqs)
        background = vectorize_texts(texts[:50])
        test_data = vectorize_texts([texts[i] for i in indices])
        id2word = {v: k for k, v in preprocessor.items()}
        explainer = shap.KernelExplainer(
            lambda x: predict_bigru_array(pytorch_model, x), background
        )
        shap_values = explainer.shap_values(test_data)
        for i, idx in enumerate(indices):
            token_vals = {}
            for j in range(test_data.shape[1]):
                tok_id = test_data[i, j]
                if tok_id != 0:
                    tok = id2word.get(tok_id, '<unk>')
                    val = shap_values[i][j]
                    if isinstance(val, np.ndarray) and val.shape[-1] == 2:
                        val = val[1]
                    token_vals[tok] = float(val)
            sorted_tokens = sorted(token_vals.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
            _plot_shap(sorted_tokens, model_name, idx, texts[idx], labels[idx], threshold, model_fn)

    else:
        raise ValueError(f'Preprocesador no reconocido para {model_name}')

In [17]:
# Seleccion de errores
def seleccionar_errores(model_fn, texts, labels, threshold=0.5, max_cases=10, batch_size=128):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        probs_batch = model_fn(batch)
        all_probs.extend(probs_batch)

    probs = np.array(all_probs)
    preds = (probs >= threshold).astype(int)
    errores_idx = [i for i,(p,y) in enumerate(zip(preds,labels)) if p!=y]
    return errores_idx[:max_cases]

In [18]:
# Seleccion de casos representativos
errores_por_modelo = {}
predicciones_por_modelo = {}
for nombre, fn in modelos.items():
    probs = fn(X_test)
    threshold = umbrales_optimos[nombre]
    preds = (probs >= threshold).astype(int)
    predicciones_por_modelo[nombre] = preds
    errores_por_modelo[nombre] = [i for i,(p,y) in enumerate(zip(preds,y_test)) if p!=y]

errores_comunes = set.intersection(*[set(v) for v in errores_por_modelo.values()])
aciertos_comunes = set.intersection(*[
    set([i for i,(p,y) in enumerate(zip(predicciones_por_modelo[n],y_test)) if p==y])
    for n in modelos.keys()
])

ejemplo_fp = next((i for i in errores_comunes if y_test[i] == 0), None)
ejemplo_fn = next((i for i in errores_comunes if y_test[i] == 1), None)
ejemplo_tp = next((i for i in aciertos_comunes if y_test[i] == 1), None)
ejemplo_tn = next((i for i in aciertos_comunes if y_test[i] == 0), None)

indices_comparativos = {
    'Falso Positivo': ejemplo_fp,
    'Falso Negativo': ejemplo_fn,
    'Acierto Positivo': ejemplo_tp,
    'Acierto Negativo': ejemplo_tn
}

for tipo, idx in indices_comparativos.items():
    if idx is None:
        print(f'No se encontro ejemplo para: {tipo}')
    else:
        print(f'{tipo}: indice {idx}')

indices_validos = [i for i in indices_comparativos.values() if i is not None]

Falso Positivo: indice 602
No se encontro ejemplo para: Falso Negativo
Acierto Positivo: indice 428
No se encontro ejemplo para: Acierto Negativo


In [19]:
# SHAP Global para todos los modelos
for nombre, fn in tqdm(modelos.items(), desc='Generando SHAP global', unit='modelo'):
    shap_global(fn, tokenizers[nombre], X_test, nombre, sample_size=20, top_n=20)

#!zip -j shap_global.zip /kaggle/working/shap_*_global.png

Generando SHAP global:   0%|          | 0/6 [00:00<?, ?modelo/s]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  25%|██▌       | 5/20 [00:23<00:17,  1.18s/it]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  35%|███▌      | 7/20 [00:39<00:59,  4.55s/it]

...

Guardado: shap_BiGRU+CNN_global.png


Generando SHAP global:  83%|████████▎ | 5/6 [09:51<01:40, 100.73s/modelo]

Guardado: shap_LogReg_global.png


Generando SHAP global: 100%|██████████| 6/6 [09:52<00:00, 98.74s/modelo] 

Guardado: shap_SVM_global.png


In [20]:
# SHAP Local en casos representativos para todos los modelos
for nombre, fn in tqdm(modelos.items(), desc='Modelos SHAP Local', unit='modelo'):
    shap_local(
        model_fn=fn,
        preprocessor=tokenizers[nombre],
        texts=X_test,
        labels=y_test,
        indices=indices_validos,
        model_name=nombre,
        threshold=umbrales_optimos[nombre]
    )

#!zip -j shap_local.zip /kaggle/working/shap_*_local_*.png

Modelos SHAP Local:   0%|          | 0/6 [00:00<?, ?modelo/s]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer: 2it [00:12, 12.61s/it]               


  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer: 2it [00:15, 15.32s/it]               
Modelos SHAP Local:  17%|█▋        | 1/6 [00:29<02:26, 29.26s/modelo]

...


PartitionExplainer explainer: 2it [00:12, 12.53s/it]               
Modelos SHAP Local:  33%|███▎      | 2/6 [00:55<01:49, 27.49s/modelo]

  0%|          | 0/2 [00:00<?, ?it/s]

Modelos SHAP Local:  50%|█████     | 3/6 [01:04<00:57, 19.12s/modelo]

  0%|          | 0/2 [00:00<?, ?it/s]

Modelos SHAP Local: 100%|██████████| 6/6 [02:09<00:00, 21.56s/modelo]


In [21]:
# Extraccion de top tokens por modelo para comparacion
def extraer_top_tokens_modelo(nombre, model_fn=None, tokenizer=None, texts=None, top_n=20,
                              vectorizer=None, vocab=None, pytorch_model=None):
    sample_texts = texts[:50]

    if tokenizer is not None and hasattr(tokenizer, 'encode'):
        def wrapped_fn(x):
            if isinstance(x, np.ndarray):
                x = x.tolist()
            return model_fn([str(t) for t in x])

        explainer = shap.Explainer(wrapped_fn, shap.maskers.Text(tokenizer))
        shap_values = explainer(sample_texts[:20])

        token_vals = {}
        for i in range(len(sample_texts[:20])):
            tokens_i = shap_values[i].data
            values_i = shap_values[i].values
            for t, v in zip(tokens_i, values_i):
                tok = t.strip()
                if tok and tok not in ['[CLS]','[SEP]','<s>','</s>','<pad>']:
                    try:
                        token_vals.setdefault(tok, []).append(float(v))
                    except:
                        pass

        token_means = {t: np.mean(v) for t,v in token_vals.items() if len(v) > 0}
        return sorted(token_means.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]

    elif vectorizer is not None:
        background = vectorizer.transform(sample_texts[:50])
        test_data = vectorizer.transform(sample_texts[:20])
        explainer = shap.LinearExplainer(logreg_model, background, feature_names=vectorizer.get_feature_names_out())
        shap_values = explainer(test_data)
        feature_names = vectorizer.get_feature_names_out()
        shap_means = np.mean(shap_values.values, axis=0)
        sorted_idx = np.argsort(np.abs(shap_means))[::-1][:top_n]
        return [(feature_names[i], shap_means[i]) for i in sorted_idx]

    elif vocab is not None and pytorch_model is not None:
        def vectorize_texts(txts, max_len=100):
            seqs = []
            for text in txts:
                tokens = text.split()
                ids = [vocab.get(tok, 0) for tok in tokens]
                ids = ids[:max_len]
                ids += [0] * (max_len - len(ids))
                seqs.append(ids)
            return np.array(seqs)

        background = vectorize_texts(texts[:50])
        test_data = vectorize_texts(texts[:20])
        explainer = shap.KernelExplainer(
            lambda x: predict_bigru_array(pytorch_model, x), background
        )
        shap_values = explainer.shap_values(test_data)
        id2word = {v: k for k, v in vocab.items()}
        token_vals = {}
        for i in range(test_data.shape[0]):
            for j in range(test_data.shape[1]):
                tok_id = test_data[i, j]
                if tok_id != 0:
                    tok = id2word.get(tok_id, '<unk>')
                    val = shap_values[i][j]
                    if isinstance(val, np.ndarray) and val.shape[-1] == 2:
                        val = val[1]
                    token_vals.setdefault(tok, []).append(float(val))

        token_means = {t: np.mean(v) for t,v in token_vals.items() if len(v) > 0}
        return sorted(token_means.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]

    else:
        raise ValueError(f'Parametros insuficientes para {nombre}')

In [22]:
# Extraer top tokens de todos los modelos
modelos_tokens = {}
for nombre in tqdm(['BETO','RoBERTa','BiGRU','BiGRU+CNN','LogReg','SVM'], desc='Extrayendo top tokens'):
    if nombre == 'BETO':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, modelos[nombre], tokenizer=beto_tokenizer, texts=X_test
        )
    elif nombre == 'RoBERTa':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, modelos[nombre], tokenizer=roberta_tokenizer, texts=X_test
        )
    elif nombre == 'BiGRU':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, texts=X_test, vocab=vocab, pytorch_model=bigru_base_model
        )
    elif nombre == 'BiGRU+CNN':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, texts=X_test, vocab=vocab, pytorch_model=bigru_cnn_model
        )
    elif nombre == 'LogReg':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, texts=X_test, vectorizer=logreg_vectorizer
        )
    elif nombre == 'SVM':
        modelos_tokens[nombre] = extraer_top_tokens_modelo(
            nombre, texts=X_test, vectorizer=svm_vectorizer
        )

Extrayendo top tokens:   0%|          | 0/6 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (860 > 512). Running this sequence through the model will result in indexing errors


  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  25%|██▌       | 5/20 [00:23<00:19,  1.29s/it]

  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer:  35%|███▌      | 7/20 [00:44<01:16,  5.90s/it]

...


PartitionExplainer explainer: 100%|██████████| 20/20 [02:57<00:00,  5.84s/it]
PartitionExplainer explainer: 21it [02:59,  8.96s/it]                        
Extrayendo top tokens:  33%|███▎      | 2/6 [06:08<12:14, 183.67s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

Extrayendo top tokens:  50%|█████     | 3/6 [07:27<06:47, 135.77s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

Extrayendo top tokens: 100%|██████████| 6/6 [17:36<00:00, 176.12s/it]


In [23]:
# Comparacion de tokens comunes entre modelos
def comparar_tokens(modelos_tokens):
    modelos_list = list(modelos_tokens.keys())
    tabla = []
    matriz = np.zeros((len(modelos_list), len(modelos_list)), dtype=int)

    for i in range(len(modelos_list)):
        for j in range(i+1, len(modelos_list)):
            inter = set([t for t,_ in modelos_tokens[modelos_list[i]]]).intersection(
                    set([t for t,_ in modelos_tokens[modelos_list[j]]]))
            tabla.append({
                'Modelo1': modelos_list[i],
                'Modelo2': modelos_list[j],
                'Tokens comunes': ', '.join(sorted(inter)) if inter else '(ninguno)'
            })
            matriz[i,j] = len(inter)
            matriz[j,i] = len(inter)

    # Tabla comparativa
    df = pd.DataFrame(tabla)
    fig, ax = plt.subplots(figsize=(14,6))
    ax.axis('off')
    ax.table(cellText=df.values, colLabels=df.columns, loc='center')
    plt.tight_layout()
    plt.savefig('comparacion_tokens_comunes.png', dpi=300)
    plt.close()

    # Heatmap
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(matriz, cmap='Blues')
    ax.set_xticks(range(len(modelos_list)))
    ax.set_yticks(range(len(modelos_list)))
    ax.set_xticklabels(modelos_list, rotation=45, ha='right')
    ax.set_yticklabels(modelos_list)
    for i in range(len(modelos_list)):
        for j in range(len(modelos_list)):
            ax.text(j, i, matriz[i,j], ha='center', va='center', color='black')
    plt.title('Numero de tokens comunes entre modelos')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig('heatmap_tokens_comunes.png', dpi=300)
    plt.close()

    # Ranking paralelo
    max_len = max(len(v) for v in modelos_tokens.values())
    data = {}
    for nombre, tokens in modelos_tokens.items():
        data[nombre] = [t for t,_ in tokens] + ['']*(max_len - len(tokens))
    df_rank = pd.DataFrame(data)
    fig, ax = plt.subplots(figsize=(14,8))
    ax.axis('off')
    ax.table(cellText=df_rank.values, colLabels=df_rank.columns, loc='center')
    plt.tight_layout()
    plt.savefig('ranking_tokens_modelos.png', dpi=300)
    plt.close()

    print('Tablas de comparacion guardadas')

comparar_tokens(modelos_tokens)
#!zip -j resultados_comparacion.zip /kaggle/working/comparacion_tokens_comunes.png /kaggle/working/heatmap_tokens_comunes.png /kaggle/working/ranking_tokens_modelos.png

Tablas de comparacion guardadas


In [24]:
# Impacto por POS tags
nlp = spacy.load('es_core_news_sm')

def agrupar_por_pos(tokens, valores):
    pos_dict = {}
    for tok, val in zip(tokens, valores):
        doc = nlp(str(tok))
        if doc:
            pos = doc[0].pos_
            pos_dict.setdefault(pos, []).append(val)
    return {pos: round(np.mean(vals), 4) for pos, vals in pos_dict.items()}

def impacto_pos_comparativo(modelos_tokens):
    resultados = {}
    for nombre, tokens_vals in modelos_tokens.items():
        tokens = [t for t,_ in tokens_vals]
        valores = [v for _,v in tokens_vals]
        resultados[nombre] = agrupar_por_pos(tokens, valores)

    df = pd.DataFrame(resultados).fillna(0).round(4)

    fig, ax = plt.subplots(figsize=(12,6))
    ax.axis('off')
    tabla = ax.table(cellText=df.values,
                     rowLabels=df.index,
                     colLabels=df.columns,
                     loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    tabla.scale(1.2, 1.2)
    plt.title('Impacto medio por POS tags - Comparacion entre modelos')
    plt.tight_layout()
    plt.savefig('impacto_pos_comparativo.png', dpi=300)
    plt.close()
    print('Tabla POS comparativa guardada')

# Individual por modelo
for nombre, tokens_vals in tqdm(modelos_tokens.items(), desc='Calculando impacto POS'):
    tokens = [t for t,_ in tokens_vals]
    valores = [v for _,v in tokens_vals]
    pos_dict = agrupar_por_pos(tokens, valores)
    df = pd.DataFrame(list(pos_dict.items()), columns=['POS','Impacto medio'])
    fig, ax = plt.subplots(figsize=(6,4))
    ax.axis('off')
    ax.table(cellText=df.values, colLabels=df.columns, loc='center')
    plt.title(f'Impacto por POS - {nombre}')
    plt.tight_layout()
    plt.savefig(f'impacto_pos_{nombre}.png', dpi=300)
    plt.close()

impacto_pos_comparativo(modelos_tokens)

#!zip -j resultados_pos.zip /kaggle/working/impacto_pos_*.png

Calculando impacto POS: 100%|██████████| 6/6 [00:02<00:00,  2.93it/s]


Tabla POS comparativa guardada


In [25]:
# Validacion con LIME
def wrap_predict_proba(fn):
    def predictor(texts):
        probs = np.array(fn(texts))
        if probs.ndim == 1:
            probs = np.vstack([1 - probs, probs]).T
        return probs
    return predictor

def lime_validacion(modelos, texts, idx=10):
    explainer = LimeTextExplainer(class_names=['Falsa','Verdadera'])
    for nombre, fn in modelos.items():
        predictor = wrap_predict_proba(fn)
        exp = explainer.explain_instance(
            texts[idx],
            predictor,
            num_features=10,
            num_samples=1000
        )
        exp.save_to_file(f'lime_{nombre}_example{idx}.html')
        fig = exp.as_pyplot_figure()
        plt.title(f'LIME - {nombre} ejemplo {idx}')
        plt.tight_layout()
        plt.savefig(f'lime_{nombre}_example{idx}.png', dpi=300)
        plt.close()

for idx in tqdm(indices_validos, desc='Validando con LIME'):
    lime_validacion(modelos, X_test, idx=idx)

#!zip -j resultados_lime.zip /kaggle/working/lime_*.png /kaggle/working/lime_*.html

Validando con LIME: 100%|██████████| 2/2 [01:47<00:00, 53.94s/it]


In [26]:
# ZIP final con todos los resultados
!zip -r resultados_completos.zip /kaggle/working/*.png /kaggle/working/*.html
print('Todos los resultados comprimidos en resultados_completos.zip')

  adding: kaggle/working/comparacion_tokens_comunes.png (deflated 56%)
  adding: kaggle/working/heatmap_tokens_comunes.png (deflated 23%)
  adding: kaggle/working/impacto_pos_BETO.png (deflated 23%)
  adding: kaggle/working/impacto_pos_BiGRU+CNN.png (deflated 23%)
  adding: kaggle/working/impacto_pos_BiGRU.png (deflated 23%)
  adding: kaggle/working/impacto_pos_comparativo.png (deflated 24%)
  adding: kaggle/working/impacto_pos_LogReg.png (deflated 23%)
  adding: kaggle/working/impacto_pos_RoBERTa.png (deflated 24%)
  adding: kaggle/working/impacto_pos_SVM.png (deflated 23%)
  adding: kaggle/working/lime_BETO_example428.png (deflated 29%)
  adding: kaggle/working/lime_BETO_example602.png (deflated 28%)
  adding: kaggle/working/lime_BiGRU+CNN_example428.png (deflated 27%)
  adding: kaggle/working/lime_BiGRU+CNN_example602.png (deflated 29%)
  adding: kaggle/working/lime_BiGRU_example428.png (deflated 30%)
  adding: kaggle/working/lime_BiGRU_example602.png (deflated 32%)
  adding: kaggle

In [27]:
# Inspeccionar registro especifico
idx = 512
print(f"Indice: {idx}")
print(f"Texto: {X_test[idx]}")
print(f"Etiqueta real: {y_test[idx]} ({'Verdadera' if y_test[idx]==1 else 'Falsa'})")

# Verificacion de la palabra number
texto = X_test[idx].lower()
if 'number' in texto.split():
    print(f"\n'number' ENCONTRADO en el texto")
else:
    print(f"\n'number' NO encontrado en el texto")

# Prediccion de cada modelo
print("\nPredicciones por modelo:")
for nombre, fn in modelos.items():
    prob = fn([X_test[idx]])[0]
    pred = int(prob >= umbrales_optimos[nombre])
    print(f"  {nombre}: prob={prob:.4f}, pred={'Verdadera' if pred==1 else 'Falsa'} (umbral={umbrales_optimos[nombre]})")

Indice: 512
Texto: Proyecto ID 2020: ¿micro-chip en la vacuna para el Covid-19? Desde 2016 se dio inicio al proyecto ID2020 que tiene como objetivo la digitalización global con datos biométricos y tecnología blockchain de todas las personas. El discurso se ha centrado en resaltar enfoques éticos y de protección a la privacidad de los datos para la identificación digital.  En este polémico proyecto participan, entre otras, The Rockefeller Foundation, Microsoft y Gavi 'The Vaccine Alliance', ésta última auspiciada por la Fundación Bill y Melinda Gates. Junto a ellos se encuentran las corporaciones Hyperledger, que promueve la tecnología blockchain; IRespond y Simprints, organizaciones dedicadas al uso de datos biométricos para la identidad digital; así como la ICC, International Computing Center de Naciones Unidas.  En mayo de 2016, en la Sede de las Naciones Unidas en Nueva York, la cumbre inaugural ID2020 reunió a más de 400 personas, entre representantes del sector público, privado, a

In [28]:
# Ver si 'number' está en el vocabulario del vectorizador
vocab_tfidf = logreg_vectorizer.vocabulary_
if 'number' in vocab_tfidf:
    idx_token = vocab_tfidf['number']
    print(f"'number' está en el vocabulario con índice {idx_token}")
    
    # Ver su valor TF-IDF en el texto 512
    X_vec = logreg_vectorizer.transform([X_test[512]])
    valor = X_vec[0, idx_token]
    print(f"Valor TF-IDF de 'number' en el texto 512: {valor}")
else:
    print("'number' no está en el vocabulario")

'number' está en el vocabulario con índice 6919
Valor TF-IDF de 'number' en el texto 512: 0.0
